## Homework

The goal of this homework is to create a simple training pipeline, use mlflow to track experiments and register best model, and a modern data workflow orchestration tool such as Mage or Prefect or others as listed in the course material under '03-orchestration'.

We'll use [the same NYC taxi dataset](https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page), the **Yellow** taxi data for March, 2023.

In [1]:
import pickle
from pathlib import Path

import mlflow
import pandas as pd
import prefect
from prefect import flow, task
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('homework3-yellow-taxi')

FILENAME = 'yellow_tripdata_2023-03.parquet'
CATEGORICAL = ['PULocationID', 'DOLocationID']

2026/10/26 22:01:28 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/26 22:01:28 INFO mlflow.store.db.utils: Updating database tables
2026/10/26 22:01:29 INFO mlflow.tracking.fluent: Experiment with name 'homework3-yellow-taxi' does not exist. Creating a new experiment.


## Question 1. Select the Tool

You can use the same tool you used when completing the module,
or choose a different one for your homework.

What's the name of the orchestrator you chose?

**Answer:** Prefect

## Question 2. Version

What's the version of the orchestrator?

In [8]:
print('Prefect version:', prefect.__version__)

Prefect version: 3.9.0


**Answer:** 3.9.0

## Question 3. Creating a pipeline

Let's read the March 2023 Yellow taxi trips data.

How many records did we load? 

- 3,003,766
- 3,203,766
- 3,403,766
- 3,603,766

(Include a print statement in your code)

In [9]:
@task
def load_data(filename: str) -> pd.DataFrame:
    df = pd.read_parquet(filename)
    print(f'records loaded: {len(df):,}')
    return df


df_raw = load_data(FILENAME)

records loaded: 3,403,766


/opt/homebrew/Cellar/python@3.11/3.11.17/Frameworks/Python.framework/Versions/3.11/lib/python3.11/logging/__init__.py:1887: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwargs)


22:09:17.471 | INFO    | Task run 'load_data' - Finished in state Completed()

11:14:48.028 | ERROR   | prefect.server.services.telemetry - Failed to send telemetry: 
12:11:40.934 | ERROR   | prefect.server.services.telemetry - Failed to send telemetry: [Errno 8] nodename nor servname provided, or not known
15:17:48.891 | ERROR   | prefect.server.services.telemetry - Failed to send telemetry: [Errno 8] nodename nor servname provided, or not known
15:39:23.591 | ERROR   | prefect.server.services.telemetry - Failed to send telemetry: [Errno 8] nodename nor servname provided, or not known


**Answer:** 3403766

## Question 4. Data preparation

Let's continue with pipeline creation.

We will use the same logic for preparing the data we used previously. 

This is what we used (adjusted for yellow dataset):

```python
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df
```

Let's apply to the data we loaded in question 3. 

What's the size of the result? 

- 2,903,766
- 3,103,766
- 3,316,216 
- 3,503,766

In [4]:
@task
def prepare_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()
    df[CATEGORICAL] = df[CATEGORICAL].astype(str)
    print(f'records after preparation: {len(df):,}')
    return df


df = prepare_data(df_raw)

records after preparation: 3,316,216


/opt/homebrew/Cellar/python@3.11/3.11.17/Frameworks/Python.framework/Versions/3.11/lib/python3.11/logging/__init__.py:1887: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwargs)


22:01:38.540 | INFO    | Task run 'prepare_data' - Finished in state Completed()

**Answer:** 3,316,216

## Question 5. Train a model

We will now train a linear regression model using the same code as in homework 1.

* Fit a dict vectorizer.
* Train a linear regression with default parameters.
* Use pick up and drop off locations separately, don't create a combination feature.

Let's now use it in the pipeline. We will need to create another transformation block, and return both the dict vectorizer and the model.

What's the intercept of the model? 

Hint: print the `intercept_` field in the code block

- 21.77
- 24.77
- 27.77
- 31.77

In [5]:
@task
def train_model(df: pd.DataFrame):
    dicts = df[CATEGORICAL].to_dict(orient='records')
    dv = DictVectorizer()
    X = dv.fit_transform(dicts)
    lr = LinearRegression()
    lr.fit(X, df.duration.values)
    print(f'intercept: {lr.intercept_:.2f}')
    return dv, lr


dv, lr = train_model(df)

intercept: 24.78


/opt/homebrew/Cellar/python@3.11/3.11.17/Frameworks/Python.framework/Versions/3.11/lib/python3.11/logging/__init__.py:1887: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwargs)


22:02:03.029 | INFO    | Task run 'train_model' - Finished in state Completed()

**Answer:** 24.78

## Question 6. Register the model 

The model is trained, so let's save it with MLFlow.

Find the logged model, and find MLModel file. What's the size of the model? (`model_size_bytes` field):

* 14,534
* 9,534
* 4,534
* 1,534

In [6]:
@task
def register_model(dv: DictVectorizer, lr: LinearRegression):
    Path('models').mkdir(exist_ok=True)
    with open('models/dv.pkl', 'wb') as f_out:
        pickle.dump(dv, f_out)

    with mlflow.start_run() as run:
        mlflow.log_param('intercept', lr.intercept_)
        mlflow.log_artifact('models/dv.pkl', artifact_path='preprocessor')
        mlflow.sklearn.log_model(
            lr, name='model', registered_model_name='yellow-taxi-linreg',
            serialization_format='cloudpickle',
        )

    # read model_size_bytes from the MLmodel file
    mlmodel_path = mlflow.artifacts.download_artifacts(f'runs:/{run.info.run_id}/model/MLmodel')
    for line in open(mlmodel_path):
        if 'model_size_bytes' in line:
            print(line.strip())


register_model(dv, lr)

2026/10/26 22:02:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'yellow-taxi-linreg'.
Created version '1' of model 'yellow-taxi-linreg'.


model_size_bytes: 4501


/opt/homebrew/Cellar/python@3.11/3.11.17/Frameworks/Python.framework/Versions/3.11/lib/python3.11/logging/__init__.py:1887: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwargs)


22:02:06.475 | INFO    | Task run 'register_model' - Finished in state Completed()

**Answer:** 4501

## Full pipeline as one Prefect flow

In [7]:
@flow(name='yellow-taxi-training')
def main(filename: str = FILENAME):
    df = load_data(filename)
    df = prepare_data(df)
    dv, lr = train_model(df)
    register_model(dv, lr)


main()

22:02:06.590 | INFO    | Flow run 'greedy-tuatara' - Beginning flow run 'greedy-tuatara' for flow 'yellow-taxi-training'

records loaded: 3,403,766


22:02:06.802 | INFO    | Task run 'load_data-ecc' - Finished in state Completed()

records after preparation: 3,316,216


22:02:09.798 | INFO    | Task run 'prepare_data-a79' - Finished in state Completed()

...

22:02:34.544 | INFO    | Task run 'train_model-7bc' - Finished in state Completed()

2026/10/26 22:02:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'yellow-taxi-linreg' already exists. Creating a new version of this model...
Created version '2' of model 'yellow-taxi-linreg'.


model_size_bytes: 4501


22:02:37.010 | INFO    | Task run 'register_model-007' - Finished in state Completed()

22:02:37.751 | INFO    | Flow run 'greedy-tuatara' - Finished in state Completed()

## Submit the results

* Submit your results here: https://courses.datatalks.club/mlops-zoomcamp-2025/homework/hw3
* If your answer doesn't match options exactly, select the closest one.